# Python Environments & Reproducible Projects
## `pip`, `conda`, `uv`, and Docker — a practical decision path

> **The goal is not to memorize commands.** The goal is to make a Python project that another
> person, machine, CI job, or container can rebuild without guesswork.

This workshop uses one small project as its running example: a reproducible Iris classification
application. We will start with the Python packaging basics, choose the right environment boundary,
then build the same project with `uv` and Docker.

### Learning outcomes

By the end, you will be able to:

- separate the roles of the interpreter, virtual environment, installer, resolver, project metadata,
  lockfile, and container;
- choose between `pip`, `conda`, and `uv` based on the dependencies you actually need;
- create a project with `pyproject.toml`, `.venv`, and `uv.lock`;
- distinguish direct dependencies from transitive dependencies;
- run commands reproducibly with `uv run` and verify a clean checkout with `uv sync --locked`;
- explain what Docker adds beyond a Python virtual environment;
- identify the files that belong in Git and the local state that does not.

### The path through this notebook

```text
1. Understand the environment boundary
   |
2. Compare pip, conda, and uv
   |
3. Build and lock one project with uv
   |
4. Reproduce it from a clean state
   |
5. Add Docker when the boundary must include the OS/runtime
```

### Workshop map

| Part | Question | Deliverable |
|---|---|---|
| 1. Concepts | What exactly must be reproducible? | A shared vocabulary |
| 2. Tool choice | Which tool fits the environment boundary? | A justified decision |
| 3. Project workflow | How do we declare, lock, and run dependencies? | A working `uv` project |
| 4. Migration and teamwork | How do we move from legacy setup to a clean project? | A migration checklist |
| 5. Containers | When is a virtual environment not enough? | A reproducible Docker image |
| 6. Practice | Can a teammate rebuild it? | A verification run |

> **Course default:** use `uv` for the exercises unless the project has a concrete reason to use
> Conda, such as a workflow dominated by native or non-Python packages. `pip` remains the universal
> foundation and is worth understanding even when `uv` is your project tool.

### Before you start

You need a terminal, Git, Python, and Docker only for the final container section. The commands are
written for macOS/Linux first; Windows PowerShell alternatives are shown where activation syntax
matters. Work from a disposable directory such as `~/workshops/python-environments`.

## 0. A real project, connected to notebooks 1 and 2

The examples below are executable. They create a small local project from the assets already built in
this workshop:

```text
LaboBots_RAG/
├── 01_hybrid_RAG_from_scratch_sections1-7.ipynb
├── 02_distributed_architecture_streamlit_litellm.ipynb
└── rag_workshop/
    ├── corpus/
    ├── chroma_db/
    └── streamlit_app.py
```

We will create a separate `./` project so that the workshop does not modify the source
materials. The project will contain copies of notebooks 1 and 2, a copy of `rag_workshop`, a Git
repository, a Python virtual environment, and explicit dependency metadata.

Run the cells in order. The bootstrap is deliberately idempotent: re-running it refreshes the copied
workshop assets but does not overwrite local secrets.

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys

ROOT = Path.cwd().resolve()
PROJECT = ROOT
SOURCE_NOTEBOOKS = [
    ROOT / "01_hybrid_RAG_from_scratch_sections1-7.ipynb",
    ROOT / "02_distributed_architecture_streamlit_litellm.ipynb",
]
SOURCE_WORKSHOP = ROOT / "rag_workshop"

missing = [path for path in [*SOURCE_NOTEBOOKS, SOURCE_WORKSHOP] if not path.exists()]
if missing:
    raise FileNotFoundError(f"Expected workshop assets are missing: {missing}")

print(f"Project root: {PROJECT}")
print(f"Using the existing {PROJECT.name} root; no duplicate project is created.")
print("Available workshop assets:")
for path in [*SOURCE_NOTEBOOKS, SOURCE_WORKSHOP / "streamlit_app.py"]:
    print(" -", path.relative_to(PROJECT))

In [ ]:
print((PROJECT / "pyproject.toml").read_text(encoding="utf-8"))
print("Using the root pyproject.toml; project metadata is not duplicated.")

In [ ]:
uv = shutil.which("uv")
venv_dir = PROJECT / ".venv"

if uv:
    subprocess.run([uv, "venv", str(venv_dir), "--python", sys.executable], check=True)
    subprocess.run([uv, "sync"], cwd=PROJECT, check=True)
    environment_python = venv_dir / ("Scripts/python.exe" if os.name == "nt" else "bin/python")
    environment_manager = "uv"
else:
    import venv
    if not venv_dir.exists():
        venv.EnvBuilder(with_pip=True).create(venv_dir)
    environment_python = venv_dir / ("Scripts/python.exe" if os.name == "nt" else "bin/python")
    subprocess.run([str(environment_python), "-m", "pip", "install", "-e", "."], cwd=PROJECT, check=True)
    environment_manager = "venv + pip"

print(f"Environment manager: {environment_manager}")
print(f"Environment Python: {environment_python}")
print(subprocess.check_output([str(environment_python), "--version"], text=True).strip())

In [ ]:
kernel_name = "labobots-rag-workshop"
subprocess.run(
    [str(environment_python), "-m", "ipykernel", "install", "--user", "--name", kernel_name,
     "--display-name", "Python (LaboBots RAG workshop)"],
    check=True,
)
print("Kernel registered: Python (LaboBots RAG workshop)")
print("In VS Code, select this kernel from the notebook kernel picker before running later cells.")

In [ ]:
git = shutil.which("git")
if git:
    print(subprocess.check_output([git, "status", "--short"], cwd=PROJECT, text=True) or "Root repository is clean.")
else:
    print("Git is not installed; skipping repository status.")

# With the pip fallback, keep an explicit snapshot for reproducibility until uv is installed.
if not uv:
    lock_snapshot = subprocess.check_output(
        [str(environment_python), "-m", "pip", "freeze"], text=True
    )
    (PROJECT / "requirements-lock.txt").write_text(lock_snapshot, encoding="utf-8")
    print("Wrote requirements-lock.txt because uv is not available on this machine.")

In [ ]:
# Validate artifacts produced by notebooks 1 and 2 without importing the optional RAG stack.
import numpy as np

corpus_dir = PROJECT / "rag_workshop" / "corpus"
chroma_dir = PROJECT / "rag_workshop" / "chroma_db"
required_files = {
    "live corpus": corpus_dir / "corpus_live_sample.json",
    "dense embeddings": corpus_dir / "embeddings.npz",
    "lexical index": corpus_dir / "lexical_weights.pkl",
    "chunks": corpus_dir / "chunks.pkl",
    "page text": corpus_dir / "page_full_text_by_url.pkl",
    "Chroma database": chroma_dir / "chroma.sqlite3",
    "Streamlit app": PROJECT / "rag_workshop" / "streamlit_app.py",
}
missing = [label for label, path in required_files.items() if not path.is_file()]
if missing:
    raise FileNotFoundError(f"Missing copied artifacts: {missing}")

with (corpus_dir / "corpus_live_sample.json").open(encoding="utf-8") as stream:
    corpus = json.load(stream)
embeddings = np.load(corpus_dir / "embeddings.npz")

print(f"Corpus records: {len(corpus) if hasattr(corpus, '__len__') else 'unknown'}")
print(f"Embedding arrays: {list(embeddings.files)}")
print(f"Chroma database: {(chroma_dir / 'chroma.sqlite3').stat().st_size:,} bytes")
print("All notebook 1/2 artifacts are present and readable.")

### 0.1 Run a real example inside the new environment

This cell writes a small application into the copied project and executes it with the Python binary
from `.venv`. The notebook kernel does not fake the result: the subprocess uses the environment we
just created.

In [ ]:
example_dir = PROJECT / "examples"
example_dir.mkdir(exist_ok=True)
example_file = example_dir / "iris_demo.py"
example_file.write_text('''from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
model = LogisticRegression(max_iter=500).fit(X_train, y_train)
accuracy = accuracy_score(y_test, model.predict(X_test))
print(f"accuracy={accuracy:.3f}")
''', encoding="utf-8")

result = subprocess.run(
    [str(environment_python), str(example_file)],
    cwd=PROJECT,
    check=True,
    capture_output=True,
    text=True,
)
print(result.stdout.strip())
accuracy = float(result.stdout.strip().split("=")[1])
assert accuracy >= 0.90, f"Unexpectedly low accuracy: {accuracy}"
print("The example ran inside the project environment.")

In [ ]:
import ast

streamlit_file = PROJECT / "rag_workshop" / "streamlit_app.py"
ast.parse(streamlit_file.read_text(encoding="utf-8"))

ignore_file = PROJECT / ".gitignore"
ignore_text = ignore_file.read_text(encoding="utf-8")
if "*.egg-info/" not in ignore_text:
    ignore_file.write_text(ignore_text + "*.egg-info/\n", encoding="utf-8")

print("Streamlit application syntax: OK")
print("Secrets file present:", (PROJECT / "rag_workshop" / ".streamlit" / "secrets.toml").exists())
print("Optional RAG dependencies are not imported in the base environment.")

### 0.2 Optional: install the full RAG stack

The copied artifacts can be inspected without downloading models. Install the heavier dependencies
only when you want to execute notebook 1's embedding/retrieval cells or launch the Streamlit client.
This step may download several hundred megabytes and BGE-M3 will still need its model weights on first
use.

In [ ]:
INSTALL_RAG_STACK = False

if INSTALL_RAG_STACK:
    subprocess.run(
        [str(environment_python), "-m", "pip", "install", "-e", ".[embeddings,secure-app]"],
        cwd=PROJECT,
        check=True,
    )
    print("Full RAG stack installed in the project environment.")
else:
    print("Skipped by default. To install it, set INSTALL_RAG_STACK = True and run this cell again.")
    print("Equivalent command:")
    print(f"  {environment_python} -m pip install -e '.[embeddings,secure-app]'")

# 1. Start with the environment boundary

The phrase **“it works on my machine”** usually means that the project boundary was left implicit.
A Python project depends on more than its `.py` files:

```text
Application
├── source code
├── Python interpreter version
├── direct Python dependencies
├── transitive dependencies
├── operating-system libraries
├── environment variables and configuration
└── external runtimes and services
```

Two developers can run the same source code and obtain different behavior when any of these layers
differs. Reproducibility is therefore a chain, not a single command:

```mermaid
flowchart LR
    S[Source code] --> M[Project metadata]
    M --> L[Resolved lockfile]
    L --> E[Python environment]
    E --> V[Verified run]
    V --> C[Optional container]
```

The first decision is not “Which command should I copy?” It is:

> **Which layers does this project need me to control?**

| Boundary you need to control | A reasonable starting point |
|---|---|
| Python packages for a small script | `venv` + `pip`, or `uv` |
| A modern Python application | `uv` project + `uv.lock` |
| Python plus substantial native/non-Python packages | `conda` or a deliberate hybrid |
| OS libraries, tools, users, and startup behavior | Docker, often with `uv` inside |

### Checkpoint 1 — name the boundary

Before continuing, write down which layers your current project depends on: Python version, Python
packages, OS libraries, external services, GPU drivers, secrets, or something else. The rest of the
notebook will make those layers explicit.

## 1.1 Vocabulary that is often confused

| Concept | Purpose | Example |
|---|---|---|
| Python interpreter | Executes Python | CPython 3.12 |
| Virtual environment | Isolates installed Python packages | `.venv/` |
| Package index | Hosts/distributes packages | PyPI |
| Installer | Installs distributions | `pip` |
| Dependency resolver | Finds compatible dependency versions | resolver in pip/uv/conda |
| Project metadata | Declares project/dependencies | `pyproject.toml` |
| Lockfile | Records a resolved dependency graph | `uv.lock` |
| Environment/package manager | Manages packages and/or environments | conda |
| Python project manager | Coordinates project, env, deps, lock/run | uv |
| Container | Isolates an application at OS/process level | Docker container |

A **virtual environment is not a container**. A `.venv` mainly isolates Python packages; a Docker image can also define the Linux userspace, OS packages, files, environment, startup command, and application runtime.

# 2. The three tools in one sentence

### `pip`
The standard Python package installer. Excellent as a fundamental, universal building block, but a complete project workflow normally also involves `venv`, dependency files/metadata, and possibly separate locking tooling.

### `conda`
A cross-language **package and environment manager**. It can manage Python itself and compiled/non-Python dependencies distributed through Conda channels.

### `uv`
A fast, modern Python package/project manager from Astral. It combines many workflows traditionally spread across `pip`, `venv`, Python-version managers, and project/lock tooling.

A useful mental model:

```text
Traditional Python:
Python installation + venv + pip + requirements/pyproject + extra tooling

Conda:
conda environment + conda packages (+ sometimes pip)

uv:
Python management + virtual environment + dependency resolver
+ project metadata + universal lockfile + command runner
```

# 3. `pip`: the baseline every Python developer should understand

`pip` installs Python distributions, usually from **PyPI**.

A classic workflow:

```bash
python -m venv .venv

# Linux/macOS
source .venv/bin/activate

# Windows PowerShell
.venv\Scripts\Activate.ps1

python -m pip install numpy pandas matplotlib
```

Why prefer `python -m pip` in teaching material? It makes explicit **which Python interpreter is launching pip**, which helps avoid the classic “I installed the package, but Python cannot import it” problem.

## 3.1 `requirements.txt`

A common file:

```text
numpy==2.2.6
pandas==2.3.2
requests==2.32.5
```

Install it with:

```bash
python -m pip install -r requirements.txt
```

A requirements file is fundamentally an **installation input format**. It can contain exact pins, ranges, URLs, editable installs, options, and more.

You may also encounter:

```bash
python -m pip freeze > requirements.txt
```

This captures the currently installed Python packages, including transitive dependencies. It can be useful, but it is not the same conceptual model as declaring only your project's intended **direct dependencies** and generating a dedicated lockfile.

## 3.2 Advantages and limitations of `pip`

### Advantages

- standard and ubiquitous in Python;
- simple mental model for installing Python packages;
- enormous PyPI ecosystem;
- works well with `venv`;
- excellent interoperability and a useful “lowest common denominator”;
- appropriate for many small scripts, libraries, teaching examples, and deployment systems.

### Limitations in a complete project workflow

- `pip` itself is primarily an installer, not an all-in-one project manager;
- Python interpreter installation/version management is separate;
- environment creation is normally handled by `venv`/`virtualenv`;
- `pip` does not itself create a project lockfile comparable to `uv.lock`;
- reproducible application workflows often require additional conventions or tools;
- it manages Python distributions, not arbitrary system packages such as CUDA runtimes or native command-line tools.

**Important nuance:** `pip` is not “bad” or obsolete. `uv` deliberately supports pip-compatible workflows because the Python packaging ecosystem built around pip/PyPI remains fundamental.

# 4. `conda`: environments beyond Python

Conda manages **environments and packages**. Its package ecosystem is not limited to Python.

Typical workflow:

```bash
conda create -n myproject python=3.12 numpy pandas
conda activate myproject
conda install scipy
```

A shareable environment can be described in YAML:

```yaml
name: myproject
channels:
  - conda-forge
dependencies:
  - python=3.12
  - numpy
  - pandas
  - scipy
```

This model is particularly useful when the environment contains substantial compiled or non-Python dependencies.

## 4.1 Where Conda is especially useful

Examples include:

- scientific/HPC stacks with native libraries;
- geospatial stacks;
- bioinformatics;
- environments mixing Python, R, command-line tools, and native libraries;
- cases where a required binary dependency is conveniently packaged in a Conda channel;
- legacy research environments already standardized on Conda.

### Strengths

- manages Python versions and environments;
- can manage non-Python dependencies;
- strong scientific ecosystem;
- mature environment export/import workflows;
- useful binary packaging model.

### Trade-offs

- a separate package ecosystem with **channels**;
- dependency solving and environment operations can feel heavier than modern uv workflows;
- channel choice/priority matters;
- mixing `pip` and Conda requires discipline;
- environment files and exact reproducibility can be platform-sensitive.

Conda's own documentation recommends care when mixing channels and, when pip is necessary inside a Conda environment, generally using Conda first and pip afterward.

# 5. `uv`: the default tool for this course

`uv` aims to provide a coherent workflow for modern Python projects.

It can handle:

- Python version installation/selection;
- virtual environments;
- package installation;
- dependency resolution;
- `pyproject.toml`;
- lockfiles;
- synchronization;
- command execution;
- development dependencies;
- pip-compatible workflows;
- running Python tools.

The key conceptual shift is:

> **Do not think “install some packages into my current environment.” Think “declare the project, resolve it, lock it, and synchronize the environment to that declaration.”**

## 5.1 Installing `uv`

Follow the official installation instructions for your platform:

- macOS/Linux: Astral provides a standalone installer and package-manager options.
- Windows: Astral provides a PowerShell installer and other options.
- It can also be installed through Python packaging channels in appropriate contexts.

Verify:

```bash
uv --version
```

> In managed school/company machines, follow your administrator's installation policy rather than blindly executing remote installation scripts.

# 6. Build the running project with `uv`

We now create the project that will be reused for the hands-on lab. Run these commands in a
terminal, one command block at a time, and inspect what changes after each step.

```bash
uv init iris-demo
cd iris-demo
```

Inspect the directory. A project will normally center around:

```text
iris-demo/
├── .python-version
├── README.md
├── pyproject.toml
└── ...
```

Add the runtime dependencies:

```bash
uv add pandas scikit-learn
```

Run Python in the project environment:

```bash
uv run python -c "import pandas, sklearn; print('Environment OK')"
```

`uv run` makes the project context explicit. It can ensure that the project environment and lock
state are up to date before running the command, so the command does not depend on which shell
happens to be active.

### What you should now see

The project should contain `pyproject.toml`, `uv.lock`, and a local `.venv/`. The lockfile is the
important new artifact: it records the resolved dependency graph, including transitive packages.

### Checkpoint 2 — inspect before moving on

Run:

```bash
uv tree
uv run python --version
git status --short
```

Confirm that you can answer three questions:

1. Which dependencies did you declare directly?
2. Which file records the resolved graph?
3. Which directory is generated local state and should not be committed?

## 6.1 What happened after `uv add`?

Three ideas matter:

### `pyproject.toml` — intent
This describes the project and its declared dependencies.

Simplified example:

```toml
[project]
name = "hello-uv"
version = "0.1.0"
requires-python = ">=3.12"
dependencies = [
    "pandas>=2.3.0",
    "requests>=2.32.0",
]
```

### `uv.lock` — resolved graph
The lockfile records a detailed resolution of the dependency graph. **Commit it to Git for applications/projects** when reproducibility matters.

### `.venv/` — installed environment
This is generated state. It is normally **not committed** to Git.

A useful rule:

```text
pyproject.toml = what the project asks for
uv.lock        = what resolution was selected
.venv/         = what is installed locally
```

# 7. Core `uv` commands

```bash
# Start a project
uv init myproject

# Add runtime dependencies
uv add numpy pandas

# Remove one
uv remove pandas

# Add development tools
uv add --dev pytest ruff

# Resolve/update the lockfile explicitly
uv lock

# Synchronize the environment with the lockfile/project
uv sync

# Run inside the project environment
uv run python main.py
uv run pytest
uv run ruff check .

# Inspect dependency tree
uv tree
```

A major usability benefit is that you often **do not need to activate `.venv` manually**. Activation is still possible, but `uv run ...` makes the intended project context explicit.

# 8. Python version management with `uv`

List/discover Python versions:

```bash
uv python list
```

Install a Python version:

```bash
uv python install 3.12
```

Pin the project:

```bash
uv python pin 3.12
```

Then:

```bash
uv run python --version
```

This removes a major source of onboarding friction: students can use one tool to manage both the project dependencies and supported Python installations.

# 9. Runtime dependencies vs development dependencies

Runtime dependencies are needed by the application itself:

```bash
uv add fastapi pydantic
```

Development dependencies support development/testing:

```bash
uv add --dev pytest ruff mypy
```

Why distinguish them?

- production images should not necessarily contain test/lint tools;
- dependency intent becomes clearer;
- CI jobs can install the groups they need;
- smaller production environments reduce attack surface and build size.

For larger projects, uv also supports **dependency groups**, allowing workflows such as linting, documentation, testing, or specialized development environments to be modeled separately.

# 10. Locking and synchronization: the heart of reproducibility

Two operations:

### Lock
Resolve dependency constraints into a concrete dependency graph.

```bash
uv lock
```

### Sync
Make the environment correspond to the resolved project.

```bash
uv sync
```

uv also integrates these operations into common project commands. For example, `uv run` normally checks the project and keeps the environment synchronized automatically.

Useful CI concepts:

```bash
uv sync --locked
uv run --locked pytest
```

`--locked` asks uv to fail rather than silently changing an out-of-date lockfile.

You may also encounter `--frozen`, which uses the lockfile without checking whether it is up to date. Understand the semantic difference before using it in automation.

# 11. `uv run`: a small command with a big architectural benefit

Compare:

### Traditional activation-oriented workflow

```bash
source .venv/bin/activate
python train.py
pytest
```

### Explicit project execution

```bash
uv run python train.py
uv run pytest
```

Benefits:

- less dependence on shell state;
- clearer CI scripts;
- fewer “wrong environment” mistakes;
- commands are self-documenting;
- easier onboarding.

For a data-science project:

```bash
uv add numpy pandas scikit-learn jupyterlab
uv run jupyter lab
```

# 12. `uvx` / tool execution

Sometimes you want to **run a Python tool**, not add it as a project dependency.

For example, depending on your workflow:

```bash
uvx ruff check .
```

This is conceptually different from:

```bash
uv add --dev ruff
uv run ruff check .
```

Use a project dependency when the tool/version is part of the project's reproducible development workflow. Use ephemeral tool execution when you intentionally want a convenient standalone invocation.

# 13. Compatibility with pip-style workflows

uv can also expose a familiar low-level interface:

```bash
uv venv
uv pip install requests
uv pip install -r requirements.txt
```

This is valuable for migration and compatibility.

But for a **new course project**, prefer the project workflow:

```bash
uv init
uv add ...
uv lock
uv sync
uv run ...
```

rather than treating uv only as “a faster pip.”

# 14. Migration example: `pip` → `uv`

Suppose an old project contains:

```text
my_app/
├── app.py
└── requirements.txt
```

with:

```text
fastapi
uvicorn
requests
```

A clean migration strategy is:

1. initialize/adopt project metadata;
2. declare the **direct** dependencies in `pyproject.toml` (using `uv add`);
3. create the lockfile;
4. test the project from a clean synchronized environment;
5. update README/CI/Docker instructions;
6. only remove legacy dependency files once no downstream consumer needs them.

Example:

```bash
cd my_app
uv init
uv add fastapi uvicorn requests
uv sync
uv run python app.py
```

For real projects, inspect version constraints and application behavior rather than mechanically copying a frozen environment into direct dependencies.

# 15. Comparison matrix

| Capability | `pip` (+ `venv`) | `conda` | `uv` |
|---|---:|---:|---:|
| Install Python packages | ✅ | ✅ | ✅ |
| PyPI-native workflow | ✅ | Partial / interoperable | ✅ |
| Create isolated environments | via `venv` | ✅ | ✅ |
| Manage Python installations | separate | ✅ | ✅ |
| Manage arbitrary non-Python packages | ❌ | ✅ | ❌* |
| `pyproject.toml` project workflow | ecosystem-based | not primary model | ✅ |
| Dedicated project lockfile | not by pip itself | environment/export mechanisms | ✅ `uv.lock` |
| Integrated command runner | ❌ | environment-oriented | ✅ `uv run` |
| Strong scientific binary ecosystem | via wheels/system deps | ✅ | via Python wheels |
| Excellent migration from pip syntax | baseline | different ecosystem | ✅ |
| Good default for modern pure-Python projects | workable | sometimes | **✅** |
| Course choice | learn concepts | know when needed | **use this** |

\* uv manages the Python project ecosystem; it is **not a general OS package manager**. System libraries still belong to the OS/container layer or another appropriate package manager.

# 16. Which tool should I use?

### Case A — small Python script
`pip + venv` works perfectly. `uv` is attractive when you want faster setup and a consistent toolchain.

### Case B — modern Python application/API
Prefer **uv**: project metadata + lockfile + synchronized environment + explicit execution.

### Case C — reusable Python library
`pyproject.toml` is central. uv is a strong development workflow, while published library metadata should express compatible dependency ranges rather than assuming consumers use your lockfile.

### Case D — data science / ML using standard Python wheels
Prefer **uv** unless a specific native dependency requires another ecosystem.

### Case E — scientific/HPC/bioinformatics stack with many native tools
**Conda may be the better environment layer**, especially when the required native packages are already well maintained in Conda channels.

### Case F — production deployment
Use uv for Python dependency/project management; consider **Docker** to package the runtime environment and application.

### Case G — CI
uv is particularly convenient:

```bash
uv sync --locked
uv run pytest
```

# 17. Common mistakes

### Mistake 1 — installing globally

```bash
pip install pandas
```

without knowing which interpreter/environment receives it.

**Better:** isolate each project.

### Mistake 2 — committing `.venv`

Do not version the environment directory. Version the files that can **reconstruct** it.

### Mistake 3 — confusing direct and transitive dependencies

If your code imports `requests`, declare `requests`. Do not manually maintain every package that `requests` itself requires.

### Mistake 4 — deleting the lockfile for an application

A lockfile is valuable reproducibility information.

### Mistake 5 — mixing Conda and pip casually

Conda and PyPI are different packaging ecosystems. If you must mix them, follow a deliberate workflow and understand which manager owns which part of the environment.

### Mistake 6 — assuming a virtual environment solves OS reproducibility

It does not capture the operating system, system libraries, users, shell tools, or external services. That is where containers become relevant.

# 18. Hands-on lab: build and verify a reproducible data project

Run these commands in a **terminal**, not blindly inside this notebook's current kernel. The goal is
to leave the terminal with a project that another person can rebuild.

```bash
uv init data-demo
cd data-demo

uv python pin 3.12
uv add pandas scikit-learn
uv add --dev pytest ruff
```

Create `main.py`:

```python
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

model = LogisticRegression(max_iter=500)
model.fit(X_train, y_train)
prediction = model.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test, prediction):.3f}")
```

Run the application and quality checks:

```bash
uv run python main.py
uv run ruff check .
uv tree
```

Then simulate a teammate's clean reconstruction. Commit the project files first, or use a fresh
clone/worktree:

```bash
uv sync --locked
uv run --locked python main.py
```

### Definition of done

You are finished when:

- `pyproject.toml` declares the direct dependencies;
- `uv.lock` is present and accepted by `uv sync --locked`;
- `.venv/` is recreated locally but is not tracked by Git;
- the application runs through `uv run`, without manual activation;
- the same commands work after removing `.venv/`;
- a teammate can understand the setup from the README and project files alone.

> **Stretch goal:** add a small `pytest` test for the expected accuracy range, then run it with
> `uv run --locked pytest`.

# 19. Exercise: reason before typing

For each situation, choose `pip`, `conda`, `uv`, or a combination. Justify the choice by naming the
boundary you need to control.

1. A one-file teaching example that imports `requests`.
2. A FastAPI application deployed through CI and Docker.
3. A bioinformatics pipeline requiring Python plus several native command-line tools available on Bioconda.
4. A machine-learning project using PyTorch and ordinary PyPI wheels.
5. A reusable Python library published to PyPI.
6. A legacy project with a 200-line `requirements.txt`.
7. A team where developers repeatedly use different Python minor versions.

### Suggested answer key

1. `pip + venv` is sufficient; `uv` is also a good low-friction choice.
2. `uv` for the Python project, Docker for the runtime boundary, and CI with `uv sync --locked`.
3. Conda, likely with a documented and deliberate `pip` step only when a package is unavailable.
4. `uv` is a strong default when the required wheels are available for the target platform.
5. A standards-based `pyproject.toml`; `uv` is useful for development, while published metadata
   should use compatible dependency ranges.
6. Migrate the direct dependencies into `pyproject.toml`, generate `uv.lock`, then verify behavior
   before deleting the legacy file.
7. Pin the supported Python version with `.python-version` and enforce it in CI.

The answer is not determined by fashion. It follows from the environment boundary and the
reproducibility guarantee the project needs.

# 20. From virtual environments to Docker

A Python virtual environment answers:

> **Which Python packages does this project use?**

A Docker image can answer a broader question:

> **Which operating-system userspace, system packages, Python runtime, Python dependencies, files,
> configuration, and startup command does this application use?**

```mermaid
flowchart TB
    H[Host machine] --> R[Container runtime]
    R --> I[Docker image]
    I --> OS[Linux userspace and system libraries]
    OS --> PY[Python runtime]
    PY --> DEPS[uv environment and locked dependencies]
    DEPS --> APP[Application and startup command]
```

The boundary is broader, but the Python dependency problem still exists inside the image. Docker
and `uv` are therefore **complementary**, not direct competitors:

```text
uv.lock  → reproducible Python dependency graph
Dockerfile → reproducible application/runtime construction
```

Use the smallest boundary that solves the problem. A virtual environment is usually enough for local
development; a container becomes valuable when OS libraries, command-line tools, service behavior,
or deployment parity also need to be controlled.

# 21. Docker vocabulary

| Term | Meaning |
|---|---|
| Dockerfile | Recipe used to build an image |
| Image | Immutable packaged filesystem/configuration |
| Container | Running instance of an image |
| Registry | Service storing images |
| Build context | Files made available during image build |
| Layer | Cached step/content in an image build |
| Volume | Persistent/external data mounted into a container |
| Port mapping | Maps a host port to a container port |

Typical lifecycle:

```bash
docker build -t myapp .
docker run --rm myapp
```

# 22. Minimal Dockerfile with Python

A conceptual example:

```dockerfile
FROM python:3.12-slim

WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY . .

CMD ["python", "main.py"]
```

This is useful for understanding Docker, but our project uses **uv**, so we can build a more coherent uv-based workflow.

# 23. Docker + `uv`

A common pattern is to copy the `uv` binary into a Python image, then synchronize from
`pyproject.toml` and `uv.lock`.

```dockerfile
FROM python:3.12-slim

# For production, pin a specific uv version or image digest.
COPY --from=ghcr.io/astral-sh/uv:latest /uv /uvx /bin/

WORKDIR /app

# Copy dependency metadata first so this layer can be cached.
COPY pyproject.toml uv.lock ./

# Install dependencies without trying to install the project yet.
RUN uv sync --locked --no-dev --no-install-project

COPY . .

# Install the project after its source code is available, if it is a package.
RUN uv sync --locked --no-dev

CMD ["uv", "run", "--no-sync", "python", "main.py"]
```

### Why copy dependency metadata before source code?

Docker caches image layers. If application source changes but dependency files do not, Docker can
reuse the dependency layer instead of reinstalling everything.

### Production checklist

- pin base-image versions or digests according to your organization's policy;
- pin the `uv` image/version;
- run as a non-root user where appropriate;
- keep secrets out of the image;
- minimize build context with `.dockerignore`;
- scan and update images and dependencies;
- decide deliberately whether the project itself is installed into the environment.

> **Compatibility note:** if the project is only a script and is not packaged, the second `uv sync`
> can be omitted. For a package project, keeping it makes imports and entry points behave as expected.

# 24. `.dockerignore`

Example:

```text
.git
.venv
__pycache__
.pytest_cache
.ruff_cache
*.pyc
.env
```

Do not send unnecessary local state, caches, virtual environments, or secrets into the Docker build context.

# 25. Docker does NOT replace dependency management

Bad mental model:

```text
Docker means dependency versions no longer matter.
```

Better mental model:

```text
uv makes the Python dependency graph reproducible.
Docker makes a broader application runtime portable/reproducible.
```

A robust project can therefore use:

```text
pyproject.toml  → project dependency intent
uv.lock         → resolved Python dependency graph
Dockerfile      → runtime/container construction
CI pipeline     → automated verification/build
```

# 26. Recommended team workflow

Use this loop for the rest of the course and for small team projects:

```bash
# 1. Clone / enter the project
cd project

# 2. Reconstruct exactly what the lockfile describes
uv sync --locked

# 3. Add a runtime dependency and update the lockfile
uv add package-name

# 4. Add a development dependency
uv add --dev pytest

# 5. Run Python and tools in the project context
uv run --locked python script.py
uv run --locked pytest
uv run --locked ruff check .

# 6. Inspect the resolved graph
uv tree
```

Files to commit:

```text
pyproject.toml
uv.lock
.python-version   # when the project intentionally pins it
source code
tests
Dockerfile        # if containerized
```

Normally do **not** commit:

```text
.venv/
__pycache__/
local secrets
.env              # when it contains secrets
```

### CI rule of thumb

Use strict commands in CI so that a missing or stale lockfile fails loudly instead of being silently
rewritten. Developers can use `uv add` and `uv lock` when changing dependencies; CI should verify the
result with `uv sync --locked` and `uv run --locked ...`.

# 27. Cheat sheet

| Task | pip/venv | conda | uv |
|---|---|---|---|
| Create env | `python -m venv .venv` | `conda create -n app python=3.12` | `uv venv` / project auto-env |
| Activate | shell-specific | `conda activate app` | often unnecessary |
| Add package | `python -m pip install X` | `conda install X` | `uv add X` |
| Dev dependency | convention/tool dependent | convention/env dependent | `uv add --dev X` |
| Install requirements | `pip install -r requirements.txt` | YAML/file workflows | `uv pip install -r requirements.txt` |
| Declare project deps | `pyproject.toml` + packaging tooling | `environment.yml` common | `uv add` → `pyproject.toml` |
| Lock | external workflow | export/explicit mechanisms | `uv lock` |
| Sync | `pip install ...` semantics | create/update env | `uv sync` |
| Run in env | activate then run | activate / `conda run` | `uv run ...` |
| Python versions | separate | conda | `uv python ...` |
| Dependency tree | ecosystem commands/tools | conda tooling | `uv tree` |

**Remember:** commands evolve. For production or teaching material maintained over time, check the current official documentation.

# 28. Final takeaways

1. **`pip` is foundational** and every Python developer should understand it.
2. **Conda remains valuable** when your environment boundary includes substantial non-Python/native dependencies.
3. **`uv` is an excellent default for modern Python project workflows** because it unifies Python versions, environments, dependency declaration, resolution, locking, synchronization, and execution.
4. Prefer **project state** (`pyproject.toml` + `uv.lock`) over an undocumented sequence of manual installations.
5. Use **`uv run`** to make project execution explicit.
6. Use **Docker when you need to control more than the Python environment**.
7. Reproducibility is not one file or one tool: it is a chain from source → dependency metadata → lockfile → runtime → automated verification.

# 29. Further reading — official documentation

Use official documentation as the source of truth because package-management tooling evolves quickly.

- **uv documentation:** `https://docs.astral.sh/uv/`
- **uv projects / locking & syncing:** `https://docs.astral.sh/uv/concepts/projects/sync/`
- **pip documentation:** `https://pip.pypa.io/`
- **pip requirements file format:** `https://pip.pypa.io/en/stable/reference/requirements-file-format/`
- **Conda documentation:** `https://docs.conda.io/`
- **Conda environments:** `https://docs.conda.io/projects/conda/en/stable/user-guide/tasks/manage-environments.html`
- **Docker Python guide:** `https://docs.docker.com/guides/python/`
- **Dockerfile reference:** `https://docs.docker.com/reference/dockerfile/`

### Suggested instructor extension

Ask students to intentionally break a dependency constraint, inspect `uv lock`/`uv tree`, recreate the project from a fresh clone, and then containerize the same project. This turns environment management from a list of commands into an understanding of **reproducibility boundaries**.